# Projet complet — Régression de prix synthétiques

Ce notebook suit question → EDA → split → baseline → modèle → évaluation → analyse d’erreurs. Les données sont simulées par une équation dans le générateur ; les résultats ne sont pas représentatifs d’un marché. Durée : 2–3 h.

## 1. Générer et comprendre le tableau

Une ligne représente un logement simulé. `price` est construit par le générateur et ne doit pas être interprété comme un prix réel.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from learning_hub.preprocessing import make_housing_data

features, target = make_housing_data(n_samples=600, random_state=42)
data = features.assign(price=target)
print(data.shape); print(data.dtypes); print(data.isna().sum()); print(data.describe(include='all').round(2))
data.head()

## 2. EDA : distributions et relations

Cherchez des incohérences de formes et d’échelles. Un graphique décrit les données générées ; il ne prouve pas que ces relations existent dans une ville réelle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(data.price, bins=25, color='slateblue')
axes[0].set(title='Prix synthétique', xlabel='Unités de prix', ylabel='Nombre')
axes[1].scatter(data.area_m2, data.price, alpha=0.45)
axes[1].set(title='Surface et prix simulés', xlabel='Surface (m²)', ylabel='Unités de prix')
plt.tight_layout(); plt.show()

## 3. Séparer train/test avant le fit

Le test final simule les observations non utilisées pour choisir les paramètres. Preprocessing et estimateur sont réunis dans une pipeline.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from learning_hub.preprocessing import make_tabular_preprocessor

numeric = ['area_m2', 'bedrooms', 'age_years', 'distance_to_center_km']
categorical = ['district']
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
models = {
    'mean baseline': Pipeline([('prep', make_tabular_preprocessor(numeric, categorical)), ('model', DummyRegressor())]),
    'random forest': Pipeline([('prep', make_tabular_preprocessor(numeric, categorical)), ('model', RandomForestRegressor(n_estimators=60, min_samples_leaf=2, random_state=42, n_jobs=1))]),
}
results = {}
for name, model in models.items():
    model.fit(x_train, y_train)
    prediction = model.predict(x_test)
    results[name] = prediction
    print(name, {'MAE': round(mean_absolute_error(y_test, prediction), 1), 'RMSE': round(mean_squared_error(y_test, prediction) ** 0.5, 1), 'R2': round(r2_score(y_test, prediction), 3)})

## 4. Lire les erreurs du meilleur candidat

La forêt exploite la relation non linéaire créée par le générateur ; la comparaison répond d’abord à « ce code suit-il le signal défini ? ». MAE et RMSE sont en unités du prix simulé.

In [ ]:
best_prediction = results['random forest']
residuals = y_test - best_prediction
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(best_prediction, residuals, alpha=0.5); axes[0].axhline(0, color='black')
axes[0].set(xlabel='Prédiction', ylabel='Résidu', title='Résidus sur test')
error_by_district = x_test[['district']].copy()
error_by_district['absolute_error'] = abs(residuals)
error_by_district.boxplot(column='absolute_error', by='district', ax=axes[1])
axes[1].set(title='Erreur par district (synthétique)', ylabel='Erreur absolue'); plt.suptitle(''); plt.tight_layout(); plt.show()
error_by_district.groupby('district').absolute_error.agg(['median', 'mean', 'count']).round(1)

## 5. Conclusion et pistes

La forêt devrait battre le baseline moyen sur le split fixe car la cible a été fabriquée à partir des variables. Cela ne valide pas les hypothèses métier ni l’estimation de biens réels.

Pistes : validation croisée répétée, baseline linéaire, comparaison des résultats par district, intervalle d’incertitude, vraie source de données avec licence vérifiée, et analyse de dérive. N’ajustez pas les hyperparamètres à partir de ce test.